# Accuracy of Our Models on Our Survey Answers

### Choose Models to Test

In [ ]:
from Training.TESTING.shap.model_wrapper import Model

models = [
    Model('age', 'regression', 'Full'),
    Model('gender', 'regression', 'Full'),
    Model('gender', 'regression', 'RoBERT'),
    Model('political', 'regression', 'Full'),
    Model('mbti', 'classification', 'Full'),
    Model('mbti', 'classification', 'LoRA')
    Model('language', 'classification', 'Full'),
    Model('language', 'classification', 'RoBERT'),
    #Model('language', 'classification', 'RoBERTLarge')
]

Initialized model age (Full)
Initialized model gender (Full)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Initialized model gender (RoBERT)
Initialized model political (Full)
Initialized model mbti (Full)
Initialized model language (Full)
Initialized model language (RoBERT)


### Testing Function for Each Trait

In [41]:
from evaluate import load
import numpy as np

df = pd.read_csv("Training/TESTING/shap/survey_filtered.csv")
accuracy = load("accuracy")

for model in models:
    trait = model.name.split(" ")[0]
    invert_labels = {v: k for k, v in model.label_map.items()} if model.label_map != None else lambda x: x

    answered = df[['text', trait]].copy()
    if trait in ['gender', 'political']:
        answered[trait] = answered[trait].map(str.lower)
    answered[trait] = answered[trait].map(invert_labels)
    answered = answered[answered[trait].notna()]

    outputs = model.predict(answered['text'])

    match model.type:
        case 'classification':
            outputs = np.argmax(outputs, axis=1)
        case 'regression':
            outputs = np.round(outputs)

    acc = accuracy.compute(predictions=outputs, references=answered[trait])

    print(f"{model.name} - accuracy: {acc}")


/home/h20/Documents/Native-Language-Identification-Author-Profiling/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")


age (Full) - accuracy: {'accuracy': 0.043478260869565216}
gender (Full) - accuracy: {'accuracy': 0.5238095238095238}
gender (RoBERT) - accuracy: {'accuracy': 0.5238095238095238}
political (Full) - accuracy: {'accuracy': 0.2631578947368421}
mbti (Full) - accuracy: {'accuracy': 0.16666666666666666}
language (Full) - accuracy: {'accuracy': 0.17391304347826086}
language (RoBERT) - accuracy: {'accuracy': 0.30434782608695654}
